# LeetCode 858 — Mirror Reflection bằng Python

Link đề: https://leetcode.com/problems/mirror-reflection/ · Độ khó: **Medium** · Chủ đề: Math, Geometry, Number Theory, GCD, LCM

Skeleton LeetCode:

```python
class Solution:
    def mirrorReflection(self, p: int, q: int) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một trong `0`, `1`, `2`). Điểm cốt lõi: **trải phẳng** (unfold) các phản xạ thành một đường thẳng trên lưới các căn phòng; đường thẳng gặp góc lưới đầu tiên ở `(a·p, b·p)` với `a = p/g`, `b = q/g`, `g = gcd(p, q)`; **tính chẵn lẻ** của `a`, `b` cho biết góc nào. Oracle: **mô phỏng tia phản xạ thật** bằng số hữu tỉ (`Fraction`), không dùng công thức.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề hình học: căn phòng vuông cạnh `p`, tia laser từ góc Tây-Nam, chạm tường Đông ở độ cao `q`, phản xạ trên gương; trả về số hiệu bộ thu (`0`, `1`, `2`) tia chạm đầu tiên.
2. Nắm công cụ: `math.gcd`, `math.lcm`, `Fraction`, tính chẵn lẻ, vòng lặp mô phỏng.
3. Đi từ **mô phỏng phản xạ (brute force)** → **trải phẳng, tìm `k` đầu tiên sao cho `k·q` chia hết `p`** → **`gcd` và chẵn lẻ (`O(log)`)** → **rút gọn cho tới khi không còn cả hai cùng chẵn**.
4. Hiểu mấu chốt: thay vì quay tia, **nhân bản căn phòng** qua gương; tia thành đường thẳng; chạm góc lưới `(a·p, b·p)`, trong đó phòng thứ `a` theo chiều ngang và thứ `b` theo chiều dọc là ảnh phản chiếu nếu chẵn/lẻ.
5. Tránh lỗi kinh điển: bỏ qua `gcd` (nên `p = q = 2` ra sai), nhầm ánh xạ chẵn/lẻ → số hiệu, nhầm vị trí các bộ thu, tưởng `q = p` cần xét riêng, mô phỏng bằng `float` bị sai số.

## 2. Đề bài & ví dụ

Có căn phòng vuông, mỗi tường là gương. Ba góc có bộ thu được đánh số `0`, `1`, `2`; góc Tây-Nam **không** có bộ thu (đó là nơi phát tia).

Cạnh phòng dài `p`. Tia laser xuất phát từ góc Tây-Nam, gặp tường Đông lần đầu ở độ cao `q` tính từ bộ thu `0`.

Trả về số hiệu bộ thu tia gặp **đầu tiên**. Đảm bảo luôn có.

Hình học: đặt hệ trục với Tây-Nam là `(0, 0)`. Từ ví dụ 1 (`p = 2, q = 1`, bộ thu `2` là nơi tia chạm khi quay lại tường Tây), ta suy ra: bộ thu `0` ở Đông-Nam `(p, 0)`, bộ thu `1` ở Đông-Bắc `(p, p)`, bộ thu `2` ở Tây-Bắc `(0, p)`.

**Ví dụ 1:** `p = 2, q = 1` → `2`.

**Ví dụ 2:** `p = 3, q = 1` → `1`.

**Ràng buộc:** `1 <= q <= p <= 1000`.

In [1]:
import time
from fractions import Fraction
from math import gcd, lcm
from typing import List

examples = [((2, 1), 2), ((3, 1), 1)]
RECEPTOR = {(1, 0): 0, (1, 1): 1, (0, 1): 2}          # (x/p, y/p) của góc -> số hiệu bộ thu
print("bộ thu 0 = (p,0) Đông-Nam; bộ thu 1 = (p,p) Đông-Bắc; bộ thu 2 = (0,p) Tây-Bắc")
for (p, q), want in examples:
    print(f"p = {p}, q = {q} -> {want}")

bộ thu 0 = (p,0) Đông-Nam; bộ thu 1 = (p,p) Đông-Bắc; bộ thu 2 = (0,p) Tây-Bắc
p = 2, q = 1 -> 2
p = 3, q = 1 -> 1


## 3. Kiến thức nền

### 3.1 Phản xạ gương và nguyên lý "trải phẳng"

Khi tia chạm tường, góc tới bằng góc phản xạ: ta **lật** thành phần vận tốc vuông góc tường. Thay vì lật tia, ta có thể **lật phòng**: đặt một bản sao đối xứng của phòng sát tường đó, và để tia đi **thẳng** sang bản sao. Mọi quỹ đạo phản xạ trong phòng gốc tương ứng một đường thẳng trong mặt phẳng xếp đầy các bản sao.

Trong lưới trải phẳng, phòng ở cột `a` (đánh số từ 0) là bản gốc lật ngang `a` lần: cột chẵn giống gốc, cột lẻ là ảnh gương theo trục x. Tương tự cho hàng `b` theo trục y.

In [2]:
# ảnh của một điểm (x, y) trong phòng gốc ở ô (a, b) của lưới trải phẳng
def unfold_point(x, y, a, b, p):
    ux = a * p + (x if a % 2 == 0 else p - x)
    uy = b * p + (y if b % 2 == 0 else p - y)
    return ux, uy


p = 5
print("điểm (1,2) trong các ô (0,0),(1,0),(0,1),(1,1):", [unfold_point(1, 2, a, b, p) for a, b in [(0, 0), (1, 0), (0, 1), (1, 1)]])
print("góc Đông-Nam (p,0) xuất hiện ở x = a*p với a lẻ, y = b*p với b chẵn:", unfold_point(p, 0, 1, 0, p), unfold_point(p, 0, 1, 2, p))

điểm (1,2) trong các ô (0,0),(1,0),(0,1),(1,1): [(1, 2), (9, 2), (1, 8), (9, 8)]
góc Đông-Nam (p,0) xuất hiện ở x = a*p với a lẻ, y = b*p với b chẵn: (5, 0) (5, 10)


### 3.2 Góc nào ứng với `(a, b)` nào?

Một góc của lưới trải phẳng có toạ độ `(a·p, b·p)`. Nó là ảnh của góc nào trong phòng gốc?

- `x = a·p`: nếu `a` lẻ thì là tường **Đông** (`x = p`), nếu `a` chẵn là tường **Tây** (`x = 0`).
- `y = b·p`: nếu `b` lẻ là tường **Bắc** (`y = p`), nếu `b` chẵn là tường **Nam** (`y = 0`).

Suy ra: `(a lẻ, b chẵn)` → Đông-Nam = `0`; `(a lẻ, b lẻ)` → Đông-Bắc = `1`; `(a chẵn, b lẻ)` → Tây-Bắc = `2`; `(a chẵn, b chẵn)` → Tây-Nam (không bộ thu, không xảy ra vì là góc phát).

In [3]:
def corner_of(a, b):
    ax = a % 2                                         # 1 = tường Đông (x = p), 0 = tường Tây (x = 0)
    by = b % 2                                         # 1 = tường Bắc (y = p), 0 = tường Nam (y = 0)
    return RECEPTOR.get((ax, by), None)                # (0,0) = Tây-Nam: None


for a in range(4):
    print([corner_of(a, b) for b in range(4)])

[None, 2, None, 2]
[0, 1, 0, 1]
[None, 2, None, 2]
[0, 1, 0, 1]


### 3.3 `gcd` và `lcm`

`gcd(p, q)` là ước chung lớn nhất; `lcm(p, q) = p·q / gcd(p, q)`. Python có `math.gcd` và (từ 3.9) `math.lcm`. Chia cả `p` và `q` cho `gcd` được hai số **nguyên tố cùng nhau**: không thể cùng chẵn.

In [4]:
print(gcd(12, 18), lcm(12, 18), gcd(12, 18) * lcm(12, 18) == 12 * 18)
print("p'/q' nguyên tố cùng nhau khi chia cho gcd:", 12 // 6, 18 // 6, gcd(2, 3))

6 36 True
p'/q' nguyên tố cùng nhau khi chia cho gcd: 2 3 1


### 3.4 `Fraction` để mô phỏng chính xác

Tia có độ dốc `q/p` hữu tỉ. Float có thể trượt sai số (`0.1 + 0.2 != 0.3`); `Fraction` chính xác tuyệt đối nên thích hợp làm oracle.

In [5]:
print(0.1 + 0.2 == 0.3, Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))

False True


## 4. Ý tưởng nền tảng / trực giác

Tia xuất phát từ `(0, 0)` hướng `(p, q)`: điểm của nó là `(t·p, t·q)`. Trong lưới trải phẳng, tia là **một đường thẳng**. Ta cần **điểm góc lưới đầu tiên** nằm trên đường thẳng (với `t > 0`): `(a·p, b·p)` với `t·p = a·p` và `t·q = b·p`, tức `t = a` và `a·q = b·p`. Số `a` nhỏ nhất thoả `a·q` chia hết `p` là `a = p / gcd(p, q)`, khi đó `b = a·q / p = q / gcd(p, q)`.

Ví dụ 1 `p = 2, q = 1`: `g = 1`, `a = 2`, `b = 1`. `(a chẵn, b lẻ)` → bộ thu `2` ✓.
Ví dụ 2 `p = 3, q = 1`: `a = 3`, `b = 1`, `(lẻ, lẻ)` → bộ thu `1` ✓.

In [6]:
for (p, q), want in examples:
    g = gcd(p, q)
    a, b = p // g, q // g
    print(f"p={p} q={q}: a = p/g = {a}, b = q/g = {b} -> bộ thu {corner_of(a, b)} (đề: {want})")
    assert corner_of(a, b) == want

p=2 q=1: a = p/g = 2, b = q/g = 1 -> bộ thu 2 (đề: 2)
p=3 q=1: a = p/g = 3, b = q/g = 1 -> bộ thu 1 (đề: 1)


## 5. Cách 1 — Brute Force: mô phỏng tia phản xạ

**Ý tưởng:** làm đúng vật lý. Giữ vị trí `(x, y)` và vận tốc `(vx, vy)`. Mỗi bước tính thời gian tới tường ngang/dọc gần nhất, nhảy tới đó; nếu chạm hai tường cùng lúc thì đã tới một **góc**; ngược lại lật thành phần vận tốc tương ứng.

In [7]:
def simulate(p, q, max_steps=10 ** 6):
    x, y = Fraction(0), Fraction(0)
    vx, vy = Fraction(p), Fraction(q)                  # hướng (p, q); độ lớn không quan trọng
    for _ in range(max_steps):
        tx = (p - x) / vx if vx > 0 else x / -vx       # thời gian tới tường x = p hoặc x = 0
        ty = (p - y) / vy if vy > 0 else y / -vy
        t = min(tx, ty)
        x, y = x + vx * t, y + vy * t
        if tx == ty:                                   # chạm góc
            return RECEPTOR[(int(x // p), int(y // p))]    # x, y chỉ là 0 hoặc p
        if tx < ty:                                    # chạm tường dọc: lật vx
            vx = -vx
        else:                                          # chạm tường ngang: lật vy
            vy = -vy
    raise RuntimeError("không chạm góc")


for (p, q), want in examples:
    assert simulate(p, q) == want
print([simulate(p, q) for (p, q), _ in examples], simulate(3, 2), simulate(5, 3))

[2, 1] 0 1


**Độ phức tạp:** số lần phản xạ trước khi tới góc là `a + b − 2 = (p + q)/g − 2` (xem mục 10), tức `O(p + q)` bước với mỗi bước `Fraction` (đắt). Bộ nhớ `O(1)`.

**Nút thắt:** ta mô phỏng từng lần phản xạ trong khi đáp án chỉ cần biết **tính chẵn lẻ** của `a` và `b`.

## 6. Ý tưởng thuật toán / bước đột phá

Trải phẳng: tia là đường thẳng `y = (q/p)·x`. Góc lưới đầu tiên là `(a·p, b·p)` với `(a, b) = (p/g, q/g)`. Vì `a` và `b` nguyên tố cùng nhau nên **không thể cùng chẵn**, và ba trường hợp còn lại ứng với ba bộ thu:

| `a = p/g` | `b = q/g` | bộ thu |
|---|---|---|
| lẻ | chẵn | `0` |
| lẻ | lẻ | `1` |
| chẵn | lẻ | `2` |

Chỉ cần `gcd` (hoặc rút gọn dần bằng cách chia 2) và xét chẵn lẻ: `O(log)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** trải phẳng bằng vòng lặp tìm `k`, **Cách 3** `gcd` và chẵn lẻ, **Cách 4** chia đôi khi cả hai cùng chẵn (không cần `gcd`).

### Cách 2 — Trải phẳng bằng vòng lặp

**Giải thích từng đoạn:**

1. Duyệt `k = 1, 2, 3, …` (`k` là số phòng đi qua theo chiều ngang, tức `a`).
2. Độ cao tia sau `k` phòng là `k·q`. Tia chạm góc khi độ cao là bội của `p`: `k·q % p == 0`.
3. Khi đó `a = k`, `b = k·q / p`; ánh xạ chẵn lẻ sang bộ thu.

In [8]:
def mirror_unfold(p, q):
    k = 1
    while (k * q) % p != 0:                             # chưa chạm góc: độ cao k*q chưa là bội của p
        k += 1
    a, b = k, k * q // p
    return corner_of(a, b)


for (p, q), want in examples:
    assert mirror_unfold(p, q) == want
print([mirror_unfold(p, q) for (p, q), _ in examples], mirror_unfold(3, 2), mirror_unfold(5, 3))

[2, 1] 0 1


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionUnfold:
    def mirrorReflection(self, p: int, q: int) -> int:
        k = 1
        while (k * q) % p != 0:
            k += 1
        a, b = k, k * q // p
        if a % 2 == 1 and b % 2 == 0:
            return 0
        if a % 2 == 1 and b % 2 == 1:
            return 1
        return 2

**Độ phức tạp:** `a = p/g ≤ p` lần lặp: `O(p)` thời gian, `O(1)` bộ nhớ. Với `p ≤ 1000` rất nhanh.

### Cách 3 — `gcd` và chẵn lẻ (đáp án chính)

**Giải thích từng đoạn:**

1. `g = gcd(p, q)`; `a = p // g`, `b = q // g`.
2. `a` lẻ và `b` chẵn → `0`; cả hai lẻ → `1`; `a` chẵn (khi đó `b` lẻ) → `2`.

In [10]:
def mirror_gcd(p, q):
    g = gcd(p, q)
    a, b = (p // g) % 2, (q // g) % 2
    if a == 1 and b == 0:
        return 0
    if a == 1 and b == 1:
        return 1
    return 2


for (p, q), want in examples:
    assert mirror_gcd(p, q) == want
print([mirror_gcd(p, q) for (p, q), _ in examples], mirror_gcd(3, 2), mirror_gcd(5, 3))

[2, 1] 0 1


In [11]:
# Code nộp LeetCode cho Cách 3
from math import gcd


class SolutionGcd:
    def mirrorReflection(self, p: int, q: int) -> int:
        g = gcd(p, q)
        p, q = (p // g) % 2, (q // g) % 2
        if p == 1 and q == 1:
            return 1
        return 0 if p == 1 else 2

**Độ phức tạp:** `O(log min(p, q))` thời gian (thuật toán Euclid), `O(1)` bộ nhớ.

### Cách 4 — Chia đôi khi cả hai cùng chẵn

**Ý tưởng:** không cần `gcd`: chừng nào `p` và `q` **cùng chẵn** thì chia cả hai cho 2 (tỉ lệ `q/p` không đổi). Khi dừng, ít nhất một số lẻ; xét chẵn lẻ y như trên. Lý do đúng: chia cho ước chung chẵn không đổi hình học; sau khi hết ước chung `2`, các ước lẻ chung chỉ thay `a, b` bằng ước của chúng với **cùng chẵn lẻ** (chia một số cho ước lẻ giữ nguyên tính chẵn lẻ).

In [12]:
def mirror_halve(p, q):
    while p % 2 == 0 and q % 2 == 0:
        p //= 2
        q //= 2
    if p % 2 == 1 and q % 2 == 1:
        return 1
    return 0 if p % 2 == 1 else 2


for (p, q), want in examples:
    assert mirror_halve(p, q) == want
print([mirror_halve(p, q) for (p, q), _ in examples], mirror_halve(3, 2), mirror_halve(5, 3), mirror_halve(4, 2), mirror_halve(8, 6))

[2, 1] 0 1 2 2


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionHalve:
    def mirrorReflection(self, p: int, q: int) -> int:
        while p % 2 == 0 and q % 2 == 0:
            p //= 2
            q //= 2
        if p % 2 == 1 and q % 2 == 1:
            return 1
        return 0 if p % 2 == 1 else 2

**Độ phức tạp:** `O(log p)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

**Trace mô phỏng** với `p = 3, q = 2`. Ray `y = (2/3)x`. Các lần chạm tường:

| bước | điểm chạm | tường | vận tốc sau |
|---|---|---|---|
| 1 | `(3, 2)` | Đông | `(−3, 2)` |
| 2 | `(3/2, 3)` | Bắc | `(−3, −2)` |
| 3 | `(0, 2)` | Tây | `(3, −2)` |
| 4 | `(3, 0)` | **góc** Đông-Nam | bộ thu `0` |

Ba lần phản xạ (`a + b − 2 = 3 + 2 − 2`), rồi tới bộ thu `0`. Hàm trace in lại đúng số liệu này:

In [14]:
def trace_simulation(p, q):
    x, y = Fraction(0), Fraction(0)
    vx, vy = Fraction(p), Fraction(q)
    print(f"p = {p}, q = {q}; xuất phát (0, 0), hướng ({p}, {q})")
    print(f"{'bước':>4} {'điểm chạm':>14} {'tường':>10} {'vận tốc sau':>14}")
    step = 0
    while True:
        tx = (p - x) / vx if vx > 0 else x / -vx
        ty = (p - y) / vy if vy > 0 else y / -vy
        t = min(tx, ty)
        x, y = x + vx * t, y + vy * t
        step += 1
        pos = f"({x}, {y})"
        if tx == ty:
            res = RECEPTOR[(int(x // p), int(y // p))]
            print(f"{step:>4} {pos:>14} {'GÓC':>10} -> bộ thu {res}")
            return res
        if tx < ty:
            wall = "Đông" if vx > 0 else "Tây"
            vx = -vx
        else:
            wall = "Bắc" if vy > 0 else "Nam"
            vy = -vy
        print(f"{step:>4} {pos:>14} {wall:>10} ({vx}, {vy})")


assert trace_simulation(3, 2) == 0
print()
assert trace_simulation(2, 1) == 2

p = 3, q = 2; xuất phát (0, 0), hướng (3, 2)
bước      điểm chạm      tường    vận tốc sau
   1         (3, 2)       Đông (-3, 2)
   2       (3/2, 3)        Bắc (-3, -2)
   3         (0, 2)        Tây (3, -2)
   4         (3, 0)        GÓC -> bộ thu 0

p = 2, q = 1; xuất phát (0, 0), hướng (2, 1)
bước      điểm chạm      tường    vận tốc sau
   1         (2, 1)       Đông (-2, 1)
   2         (0, 2)        GÓC -> bộ thu 2


**Trace Cách 3** với `p = 3, q = 2`: `g = 1`, `a = 3` (lẻ), `b = 2` (chẵn) → bộ thu `0` ✓. Với `p = 6, q = 4`: `g = 2`, `a = 3`, `b = 2` → `0`. Với `p = 4, q = 4`: `g = 4`, `a = b = 1` → `1`.

In [15]:
def trace_gcd(p, q):
    g = gcd(p, q)
    a, b = p // g, q // g
    res = corner_of(a, b)
    print(f"p={p} q={q}: g={g}, a=p/g={a} ({'lẻ' if a % 2 else 'chẵn'}), b=q/g={b} ({'lẻ' if b % 2 else 'chẵn'}) -> bộ thu {res}")
    return res


for pq in [(3, 2), (6, 4), (4, 4), (2, 1), (3, 1), (5, 4)]:
    assert trace_gcd(*pq) == simulate(*pq)

p=3 q=2: g=1, a=p/g=3 (lẻ), b=q/g=2 (chẵn) -> bộ thu 0
p=6 q=4: g=2, a=p/g=3 (lẻ), b=q/g=2 (chẵn) -> bộ thu 0
p=4 q=4: g=4, a=p/g=1 (lẻ), b=q/g=1 (lẻ) -> bộ thu 1
p=2 q=1: g=1, a=p/g=2 (chẵn), b=q/g=1 (lẻ) -> bộ thu 2
p=3 q=1: g=1, a=p/g=3 (lẻ), b=q/g=1 (lẻ) -> bộ thu 1
p=5 q=4: g=1, a=p/g=5 (lẻ), b=q/g=4 (chẵn) -> bộ thu 0


## 9. Code nộp LeetCode

Chọn **`gcd` và chẵn lẻ**. Giữ nguyên `class Solution`, `mirrorReflection(self, p, q)`:

In [16]:
from math import gcd


class Solution:
    def mirrorReflection(self, p: int, q: int) -> int:
        g = gcd(p, q)
        p, q = (p // g) % 2, (q // g) % 2                 # chẵn lẻ của a = p/g và b = q/g
        if p == 1 and q == 1:
            return 1                                      # Đông-Bắc
        return 0 if p == 1 else 2                         # a lẻ, b chẵn -> 0; a chẵn -> 2


print([Solution().mirrorReflection(*pq) for pq, _ in examples])    # [2, 1]
assert all(Solution().mirrorReflection(*pq) == want for pq, want in examples)

[2, 1]


## 10. Vì sao đúng

**(a) Tia là đường thẳng trong lưới trải phẳng.** Phản xạ trên tường tương đương đi thẳng sang bản sao lật của phòng. Do đó vị trí tia ở thời điểm `t` (đơn vị: `t = 1` ứng với đi ngang một cạnh `p`) là `(t·p, t·q)` trong mặt phẳng trải phẳng, rồi gấp lại về phòng gốc.

**(b) Tia chạm góc khi cả hai toạ độ là bội của `p`.** Góc của phòng nào cũng nằm tại điểm `(a·p, b·p)` với `a, b` nguyên. Tia ở `(t·p, t·q)`; cần `t = a` và `a·q = b·p`. Điều kiện `a·q ≡ 0 (mod p)` nghĩa là `p | a·q`; chia cho `g`: `(p/g) | a·(q/g)`, mà `gcd(p/g, q/g) = 1` nên `(p/g) | a`. Nhỏ nhất: `a = p/g`, `b = q/g`.

**(c) Góc nào?** Nhìn mục 3.2: `a` lẻ → tường Đông (`x = p`), `a` chẵn → tường Tây; `b` lẻ → Bắc, `b` chẵn → Nam. `(a, b)` nguyên tố cùng nhau (đã chia `g`) nên không cùng chẵn (không bao giờ về góc Tây-Nam, khớp lời đề "chắc chắn gặp bộ thu"). Ba trường hợp còn lại cho bộ thu `0`, `1`, `2`.

**(d) Số lần phản xạ.** Trên đường tới góc, tia cắt `a − 1` đường dọc và `b − 1` đường ngang, tức `a + b − 2` lần phản xạ.

**(e) Cách 4.** Chia cả hai cho `2` khi cùng chẵn: tỉ số `q/p` giữ nguyên nên `(a, b)` giữ nguyên. Sau khi dừng, `p', q'` không cùng chẵn: `p' = a·h`, `q' = b·h` với `h = gcd(p', q')` lẻ (vì `gcd` chứa `2` mới có thể làm cả hai chẵn). Nhân một số với số lẻ `h` không đổi tính chẵn lẻ, nên chẵn lẻ của `(p', q')` bằng của `(a, b)`.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **mô phỏng phản xạ** trên mọi `1 ≤ q ≤ p ≤ 120` (mục 12D) và khớp nhau trên toàn bộ `500 500` cặp của ràng buộc (mục 12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng phản xạ | `O((p+q)/g)` với `Fraction` | `O(1)` | oracle |
| 2. Trải phẳng, vòng lặp `k` | `O(p/g)` | `O(1)` | dễ hiểu |
| 3. `gcd` + chẵn lẻ | `O(log min(p,q))` | `O(1)` | **đáp án chính** |
| 4. Chia đôi | `O(log p)` | `O(1)` | không cần `gcd` |

Đo thực tế:

In [17]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(20):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / 20)
    return best


pairs = [(1000, 999), (997, 1), (1000, 500), (729, 243)]
for p, q in pairs:
    row = [f"{name} {bench(fn, (p, q)) * 1e6:9.1f} us" for name, fn in
           [("sim", simulate), ("unfold", mirror_unfold), ("gcd", mirror_gcd), ("halve", mirror_halve)]]
    print(f"p={p:4d} q={q:4d}: " + " | ".join(row))

p=1000 q= 999: sim   13423.7 us | unfold      50.7 us | gcd       0.1 us | halve       0.1 us


p= 997 q=   1: sim    6962.3 us | unfold      89.6 us | gcd       0.3 us | halve       0.1 us
p=1000 q= 500: sim      24.8 us | unfold       0.5 us | gcd       0.3 us | halve       0.4 us
p= 729 q= 243: sim      34.6 us | unfold       0.6 us | gcd       0.3 us | halve       0.2 us


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi `1 ≤ q ≤ p ≤ 120` với oracle mô phỏng phản xạ, (E) ngẫu nhiên có seed và tính chất đối xứng, (F) toàn bộ ràng buộc `500 500` cặp.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "unfold": mirror_unfold,
    "gcd": mirror_gcd,
    "halve": mirror_halve,
    "Solution": lambda p, q: Solution().mirrorReflection(p, q),
    "SolutionUnfold": lambda p, q: SolutionUnfold().mirrorReflection(p, q),
    "SolutionGcd": lambda p, q: SolutionGcd().mirrorReflection(p, q),
    "SolutionHalve": lambda p, q: SolutionHalve().mirrorReflection(p, q),
}


def check(p, q, want=None):
    if want is None:
        want = simulate(p, q)
    for name, fn in IMPLS.items():
        assert fn(p, q) == want, (name, p, q, fn(p, q), want)


for (p, q), want in examples:
    check(p, q, want)
    assert simulate(p, q) == want
print("ví dụ đề: oracle mô phỏng và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle mô phỏng và 7 cài đặt đúng


### B. Edge case

- `p = q = 1`: tia đi thẳng tới Đông-Bắc → `1`.
- `q = p` bất kỳ → `1` (đường chéo).
- `q = 1`: số lẻ/chẵn của `p` quyết định.
- `p = 2^k`, `q = 1` → `2`; `q = p/2` → `2`? (kiểm bằng oracle).
- `p`, `q` nguyên tố cùng nhau và lớn: `(1000, 999)`, `(999, 998)`.
- `(1000, 1000)`, `(1000, 1)`.

In [19]:
cases = [
    (1, 1, 1), (2, 2, 1), (1000, 1000, 1), (3, 3, 1),
    (2, 1, 2), (4, 1, 2), (8, 1, 2), (3, 1, 1), (1000, 1, 2), (999, 1, 1),
    (3, 2, 0), (5, 2, 0), (999, 998, 0), (1000, 999, 2),
    (6, 4, 0), (4, 2, 2), (8, 6, 2),
]
for p, q, want in cases:
    check(p, q, want)
print(len(cases), "edge case: mô phỏng và mọi cài đặt cùng đúng")

17 edge case: mô phỏng và mọi cài đặt cùng đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_no_gcd(p, q):                                  # xét chẵn lẻ của p, q gốc, không chia gcd
    a, b = p % 2, q % 2
    if a == 1 and b == 1:
        return 1
    return 0 if a == 1 else 2


def bug_swap_labels(p, q):                             # nhầm 0 <-> 2
    r = mirror_gcd(p, q)
    return {0: 2, 1: 1, 2: 0}[r]


def bug_swap_parity(p, q):                             # nhầm vai trò a, b (b = p/g, a = q/g)
    g = gcd(p, q)
    a, b = (q // g) % 2, (p // g) % 2
    if a == 1 and b == 1:
        return 1
    return 0 if a == 1 else 2


def bug_only_p_parity(p, q):                           # chỉ nhìn p
    return 1 if p % 2 == 1 else 2


def bug_float(p, q):                                   # mô phỏng bằng float: sai số khiến không nhận ra góc
    x = y = 0.0
    vx, vy = float(p), float(q)
    for _ in range(2 * (p + q) + 5):                   # đủ số bước nếu tính chính xác
        tx = (p - x) / vx if vx > 0 else x / -vx
        ty = (p - y) / vy if vy > 0 else y / -vy
        t = min(tx, ty)
        x, y = x + vx * t, y + vy * t
        if tx == ty:
            return RECEPTOR.get((round(x / p), round(y / p)), -1)
        if tx < ty:
            vx = -vx
        else:
            vy = -vy
    return -1                                          # không bao giờ thấy tx == ty chính xác (hoặc sai góc)


BUGS = [bug_no_gcd, bug_swap_labels, bug_swap_parity, bug_only_p_parity, bug_float]
allpq = [(p, q) for p in range(1, 41) for q in range(1, p + 1)]
for bug in BUGS:
    bad = next(((p, q) for p, q in allpq if bug(p, q) != mirror_gcd(p, q)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:17s} sai trên p,q = {bad}: ra {bug(*bad)}, đúng {mirror_gcd(*bad)}")

bug_no_gcd        sai trên p,q = (2, 2): ra 2, đúng 1
bug_swap_labels   sai trên p,q = (2, 1): ra 0, đúng 2
bug_swap_parity   sai trên p,q = (2, 1): ra 0, đúng 2
bug_only_p_parity sai trên p,q = (2, 2): ra 2, đúng 1
bug_float         sai trên p,q = (7, 5): ra -1, đúng 1


### D. Vét cạn mọi cặp nhỏ

Mọi `1 ≤ q ≤ p ≤ 120` (`7 260` cặp). Oracle: mô phỏng phản xạ `Fraction`. Mọi cài đặt phải khớp.

In [21]:
t0 = time.perf_counter()
cnt = 0
for p in range(1, 121):
    for q in range(1, p + 1):
        want = simulate(p, q)
        for name, fn in IMPLS.items():
            assert fn(p, q) == want, (name, p, q)
        cnt += 1
print(cnt, "cặp: 7 cài đặt khớp oracle mô phỏng phản xạ;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 7260

7260 cặp: 7 cài đặt khớp oracle mô phỏng phản xạ; 5.2 s


### E. Ngẫu nhiên có seed và tính chất

Chọn ngẫu nhiên `(p, q)` tới `1000` và so với mô phỏng; kiểm tra thêm các **tính chất**: (1) nhân cả `p, q` cho `m` không đổi đáp án (cùng tia); (2) `q = p` luôn cho `1`; (3) số lần phản xạ `(p+q)/g − 2` bằng số bước mô phỏng trừ 1.

In [22]:
import random

rng = random.Random(20240858)
for _ in range(1500):
    p = rng.randint(1, 1000)
    q = rng.randint(1, p)
    check(p, q)
    m = rng.randint(1, 1000 // p)
    assert Solution().mirrorReflection(p * m, q * m) == Solution().mirrorReflection(p, q)
print("1500 cặp ngẫu nhiên khớp mô phỏng; tính chất nhân tỉ lệ đúng")
assert all(Solution().mirrorReflection(p, p) == 1 for p in range(1, 1001))


def count_hits(p, q):                                  # số lần chạm tường (không tính lần chạm góc) khi mô phỏng
    x = y = Fraction(0)
    vx, vy = Fraction(p), Fraction(q)
    hits = 0
    while True:
        tx = (p - x) / vx if vx > 0 else x / -vx
        ty = (p - y) / vy if vy > 0 else y / -vy
        t = min(tx, ty)
        x, y = x + vx * t, y + vy * t
        if tx == ty:
            return hits
        hits += 1
        if tx < ty:
            vx = -vx
        else:
            vy = -vy


for _ in range(300):
    p = rng.randint(1, 60)
    q = rng.randint(1, p)
    g = gcd(p, q)
    assert count_hits(p, q) == (p + q) // g - 2, (p, q)
print("số lần phản xạ = (p + q)/g - 2 đúng trên 300 cặp")

1500 cặp ngẫu nhiên khớp mô phỏng; tính chất nhân tỉ lệ đúng
số lần phản xạ = (p + q)/g - 2 đúng trên 300 cặp


### F. Toàn bộ ràng buộc

Mọi `1 ≤ q ≤ p ≤ 1000` (`500 500` cặp): so `gcd`, `halve` và công thức `lcm` (`a = lcm/q`, `b = lcm/p`) — nhanh, không dùng mô phỏng; riêng cách trải phẳng chạy tới `p ≤ 200` để tiết kiệm thời gian.

In [23]:
def mirror_lcm(p, q):
    L = lcm(p, q)
    a, b = L // q, L // p                              # a = p/g, b = q/g
    return corner_of(a, b)


t0 = time.perf_counter()
cnt = 0
dist = {0: 0, 1: 0, 2: 0}
for p in range(1, 1001):
    for q in range(1, p + 1):
        r = mirror_gcd(p, q)
        assert r == mirror_halve(p, q) == mirror_lcm(p, q), (p, q)
        if p <= 200:
            assert r == mirror_unfold(p, q), (p, q)
        dist[r] += 1
        cnt += 1
print(cnt, "cặp: gcd = halve = lcm (và = unfold khi p <= 200);", round(time.perf_counter() - t0, 1), "s")
print("phân bố đáp án:", dist)
assert cnt == 500500 and sum(dist.values()) == cnt

500500 cặp: gcd = halve = lcm (và = unfold khi p <= 200); 0.8 s
phân bố đáp án: {0: 166198, 1: 167198, 2: 167104}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ qua `gcd`

Chỉ xét chẵn lẻ của `p` và `q` gốc. Khi `p, q` có ước chung (đặc biệt cả hai chẵn) thì sai.

In [24]:
print("p=q=2 ->", bug_no_gcd(2, 2), "| đúng:", Solution().mirrorReflection(2, 2), "| mô phỏng:", simulate(2, 2))
assert bug_no_gcd(2, 2) == 2 and Solution().mirrorReflection(2, 2) == 1

p=q=2 -> 2 | đúng: 1 | mô phỏng: 1


**Vì sao sai:** `p = q = 2` là đường chéo, tia chạm Đông-Bắc (`1`), còn chẵn lẻ gốc cho `2`. **Đúng:** chia cho `gcd` trước (hoặc chia đôi tới khi không cùng chẵn).

### Lỗi 2: nhầm ánh xạ số hiệu bộ thu

Bộ thu `0` ở **Đông-Nam**, `2` ở **Tây-Bắc**. Đổi chỗ hai số cho đáp án ngược.

In [25]:
print("p=3,q=2 ->", bug_swap_labels(3, 2), "| đúng:", Solution().mirrorReflection(3, 2))
assert bug_swap_labels(3, 2) == 2 and Solution().mirrorReflection(3, 2) == 0

p=3,q=2 -> 2 | đúng: 0


**Vì sao sai:** `a` lẻ nghĩa tường Đông, `b` chẵn nghĩa Nam → góc Đông-Nam = bộ thu `0`. **Đúng:** theo bảng mục 6.

### Lỗi 3: nhầm vai trò `a` và `b`

`a = p/g` đếm số phòng **ngang**; `b = q/g` đếm số phòng **dọc**. Đảo hai giá trị đổi `0` thành `2` và ngược lại.

In [26]:
print("p=3,q=2 ->", bug_swap_parity(3, 2), "| đúng:", Solution().mirrorReflection(3, 2))
assert bug_swap_parity(3, 2) == 2

p=3,q=2 -> 2 | đúng: 0


**Vì sao sai:** tia đi `p` theo x cho mỗi `q` theo y, nên số phòng ngang là `p/g`. **Đúng:** `a = p // g` là thành phần quyết định tường Đông/Tây.

### Lỗi 4: chỉ nhìn chẵn lẻ của `p`

Bỏ qua `q`. Trường hợp `p` lẻ, `q` chẵn là bộ thu `0`, không phải `1`.

In [27]:
print("p=3,q=2 ->", bug_only_p_parity(3, 2), "| đúng:", Solution().mirrorReflection(3, 2))
assert bug_only_p_parity(3, 2) == 1

p=3,q=2 -> 1 | đúng: 0


**Vì sao sai:** cần **cả hai** chẵn lẻ. **Đúng:** xét `(a mod 2, b mod 2)`.

### Lỗi 5: mô phỏng bằng `float`

Sai số khiến `tx == ty` không bao giờ đúng, không nhận ra góc (hoặc nhận nhầm).

In [28]:
float_bad = [(p, q) for p in range(1, 31) for q in range(1, p + 1) if bug_float(p, q) != mirror_gcd(p, q)]
print(len(float_bad), "cặp (p, q) với p <= 30 mà bản float ra sai/không kết thúc, ví dụ:", float_bad[:5])
assert float_bad

73 cặp (p, q) với p <= 30 mà bản float ra sai/không kết thúc, ví dụ: [(7, 5), (7, 6), (9, 7), (10, 9), (11, 9)]


**Vì sao sai:** `(p - x)/vx` là số thực không biểu diễn chính xác. **Đúng:** dùng số nguyên/`Fraction` hoặc bỏ mô phỏng, dùng `gcd`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tia phản xạ trên tường/gương hình chữ nhật" → **trải phẳng** (nhân bản không gian qua gương), biến phản xạ thành đường thẳng; sau đó dùng **số học** (`gcd`/`lcm`, chẵn lẻ) trên lưới.

**Mẹo:**

1. Vẽ lưới các phòng bản sao và đường thẳng; đếm góc lưới đầu tiên.
2. Tính chẵn lẻ của toạ độ lưới cho biết tường/góc thật.
3. Chia cho `gcd` để các số nguyên tố cùng nhau (loại trường hợp "cùng chẵn").
4. Oracle độc lập: mô phỏng bằng `Fraction`.
5. Không dùng `float` cho kiểm tra bằng nhau.

**Họ bài cùng pattern:** Bouncing Ball / Light ray trong chữ nhật, Water and Jug Problem (365), Count Lattice Points (đếm điểm nguyên trên đường), Robot Bounded In Circle (1041), Rotate Function.

## 15. Biến thể

1. **Hình chữ nhật `w × h`:** tia hướng `(w, d)`; góc lưới đầu tiên `(a·w, b·h)` với `a = h/g'`, `b = ...` (dùng `lcm(w·?, ...)`).
2. **Số lần phản xạ trước khi tới bộ thu:** `(p + q)/g − 2`.
3. **Điểm bắt đầu khác** (không phải góc).
4. **Nhiều tia / mục tiêu là điểm bất kỳ.**
5. **Tọa độ chạm tường thứ `k`.**

Cài biến thể 1 (chữ nhật `W × H`, tia hướng `(W, d)` tới tường Đông ở độ cao `d`; góc: `0` = Đông-Nam `(W, 0)`, `1` = Đông-Bắc `(W, H)`, `2` = Tây-Bắc `(0, H)`) và đối chiếu mô phỏng:

In [29]:
def rect_sim(W, H, d):
    x = y = Fraction(0)
    vx, vy = Fraction(W), Fraction(d)
    while True:
        tx = (W - x) / vx if vx > 0 else x / -vx
        ty = (H - y) / vy if vy > 0 else y / -vy
        t = min(tx, ty)
        x, y = x + vx * t, y + vy * t
        if tx == ty:
            return {(W, 0): 0, (W, H): 1, (0, H): 2}[(int(x), int(y))]
        if tx < ty:
            vx = -vx
        else:
            vy = -vy


def rect_formula(W, H, d):
    # tia (t*W, t*d); góc khi t*W = a*W và t*d = b*H -> a*d = b*H; a nhỏ nhất = H/gcd(H, d), b = d/gcd(H, d)
    g = gcd(H, d)
    a, b = H // g, d // g
    return corner_of(a, b)


for W in range(1, 15):
    for H in range(1, 15):
        for d in range(1, H + 1):
            assert rect_sim(W, H, d) == rect_formula(W, H, d), (W, H, d)
assert all(rect_formula(p, p, q) == Solution().mirrorReflection(p, q) for p in range(1, 30) for q in range(1, p + 1))
print("phòng chữ nhật: công thức a = H/gcd(H,d), b = d/gcd(H,d) khớp mô phỏng; W = H quay về bài gốc")

phòng chữ nhật: công thức a = H/gcd(H,d), b = d/gcd(H,d) khớp mô phỏng; W = H quay về bài gốc


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | tính va chạm từng bước (`Fraction`) | `O((p+q)/g)` | `O(1)` | oracle |
| 2. Trải phẳng, vòng `k` | tìm `k` nhỏ nhất sao cho `k·q % p == 0` | `O(p/g)` | `O(1)` | dễ hiểu |
| 3. `gcd` + chẵn lẻ | `a = p/g`, `b = q/g` | `O(log)` | `O(1)` | **đáp án chính** |
| 4. Chia đôi | bỏ ước chung `2` | `O(log)` | `O(1)` | không cần `gcd` |

Cả bốn khớp oracle mô phỏng trên 7 260 cặp và khớp nhau trên toàn bộ 500 500 cặp (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().mirrorReflection(4, 2), Solution().mirrorReflection(5, 5), Solution().mirrorReflection(7, 3), Solution().mirrorReflection(6, 4))

2 1 1 0


### Bài 2 (tính tay)

`p = 6, q = 4`: (a) `g`, `a`, `b`? (b) bộ thu nào? (c) có bao nhiêu lần phản xạ trước khi tới bộ thu? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện `gcd` và chẵn lẻ:

In [31]:
def gcd_todo(p, q):
    g = 0           # TODO 1: gcd của p và q
    a, b = 0, 0     # TODO 2: chẵn lẻ của p // g và q // g
    return 0        # TODO 3: ánh xạ (a, b) -> 0 / 1 / 2


# Khi làm xong, bỏ comment để kiểm tra:
# assert [gcd_todo(p, q) for (p, q), _ in examples] == [2, 1] and gcd_todo(3, 2) == 0

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **cách chia đôi**:

In [32]:
class MySolution:
    def mirrorReflection(self, p, q):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().mirrorReflection(*pq) == w for pq, w in examples)

### Bài 5 (mini challenge): bộ thu cuối cùng và đếm số lần chạm tường

Viết `wall_hits(p, q)` trả số lần tia chạm tường (không tính bộ thu) trước khi tới bộ thu, theo công thức, không mô phỏng. So với `count_hits` (mô phỏng) ở mục 12E.

In [33]:
def wall_hits(p, q):
    pass            # TODO


# Khi làm xong:
# assert wall_hits(3, 2) == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: rút gọn `p, q` bằng `gcd` rồi xét chẵn lẻ.
- Mức 2: `(4, 2)` → `(2, 1)`: `a` chẵn → `2`; `(5, 5)` → `(1, 1)` → `1`; `(7, 3)` → `a` lẻ, `b` lẻ → `1`; `(6, 4)` → `(3, 2)` → `0`.
- Mức 3: `2 1 1 0`.

**Bài 2**

- Mức 1: `g = gcd(6, 4) = 2`.
- Mức 2: `a = 3` (lẻ), `b = 2` (chẵn) → bộ thu `0`.
- Mức 3: số lần phản xạ `a + b − 2 = 3`.

**Bài 3**

- Mức 1: `math.gcd`.
- Mức 2: `a, b = (p // g) % 2, (q // g) % 2`.
- Mức 3: `a` và `b` đều lẻ → `1`; `a` lẻ → `0`; còn lại `2`.

**Bài 4**

- Mức 1: `while p % 2 == 0 and q % 2 == 0`.
- Mức 2: chia đôi cả hai.
- Mức 3: xem Cách 4.

**Bài 5**

- Mức 1: tia cắt `a − 1` đường dọc và `b − 1` đường ngang.
- Mức 2: `a + b − 2`.
- Mức 3: `(p + q) // gcd(p, q) - 2`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
assert [Solution().mirrorReflection(*pq) for pq in [(4, 2), (5, 5), (7, 3), (6, 4)]] == [2, 1, 1, 0]
assert [simulate(*pq) for pq in [(4, 2), (5, 5), (7, 3), (6, 4)]] == [2, 1, 1, 0]
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) `g = 2`, `a = 3`, `b = 2`. (b) Bộ thu `0`. (c) `a + b − 2 = 3` lần phản xạ.

In [35]:
g = gcd(6, 4)
a, b = 6 // g, 4 // g
assert (g, a, b) == (2, 3, 2)
assert Solution().mirrorReflection(6, 4) == 0 == simulate(6, 4)
assert count_hits(6, 4) == a + b - 2 == 3
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [36]:
def gcd_answer(p, q):
    g = gcd(p, q)
    a, b = (p // g) % 2, (q // g) % 2
    if a and b:
        return 1
    return 0 if a else 2


assert all(gcd_answer(p, q) == simulate(p, q) for p in range(1, 40) for q in range(1, p + 1))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def mirrorReflection(self, p, q):
        while p % 2 == 0 and q % 2 == 0:
            p //= 2
            q //= 2
        if p % 2 and q % 2:
            return 1
        return 0 if p % 2 else 2


assert all(MySolutionAnswer().mirrorReflection(p, q) == simulate(p, q) for p in range(1, 40) for q in range(1, p + 1))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def wall_hits_answer(p, q):
    return (p + q) // gcd(p, q) - 2


assert wall_hits_answer(3, 2) == 3 == count_hits(3, 2)
assert all(wall_hits_answer(p, q) == count_hits(p, q) for p in range(1, 50) for q in range(1, p + 1))
print("Bài 5 OK: công thức khớp mô phỏng với p <= 49")

Bài 5 OK: công thức khớp mô phỏng với p <= 49


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tia phản xạ trong phòng vuông → trải phẳng, `gcd`, chẵn lẻ):**

```python
from math import gcd

class Solution:
    def mirrorReflection(self, p: int, q: int) -> int:
        g = gcd(p, q)
        p, q = (p // g) % 2, (q // g) % 2
        if p == 1 and q == 1:
            return 1
        return 0 if p == 1 else 2
```

**Checklist trước khi submit:**

- [ ] Đã chia cho `gcd(p, q)` (hoặc chia đôi tới khi không cùng chẵn) trước khi xét chẵn lẻ?
- [ ] Hiểu: `a = p/g` (phòng ngang), `b = q/g` (phòng dọc); `(a lẻ, b chẵn) → 0`, `(lẻ, lẻ) → 1`, `(chẵn, lẻ) → 2`?
- [ ] Bộ thu `0` Đông-Nam, `1` Đông-Bắc, `2` Tây-Bắc?
- [ ] Không dùng `float` để so sánh bằng khi mô phỏng?
- [ ] Đã thử `(2,1)→2`, `(3,1)→1`, `(2,2)→1`, `(3,2)→0`?
- [ ] Đúng tên `class Solution`, `mirrorReflection(self, p, q)`, không in debug.